In [ ]:
# 1. Implement a simple text classification model using LSTM in Keras

import numpy as np
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

# Sample data
texts = ["I love natural language processing", "Deep learning is amazing", "Keras makes life easy"]
labels = [1, 1, 0]

# Tokenization
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index

data = pad_sequences(sequences, maxlen=10)

# Build LSTM model
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=32, input_length=10))
model.add(LSTM(32))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(data, np.array(labels), epochs=10, batch_size=2)


In [ ]:
# 2. Generate sequences of text using a Recurrent Neural Network (RNN)

from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical

# Sample text data
text = "The quick brown fox jumps over the lazy dog"
text = text.lower()
chars = sorted(list(set(text)))
char_indices = {c: i for i, c in enumerate(chars)}
indices_char = {i: c for i, c in enumerate(chars)}

maxlen = 10
step = 1
sentences = []
next_chars = []

for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i + maxlen])

X = np.zeros((len(sentences), maxlen, len(chars)), dtype=np.bool)
y = np.zeros((len(sentences), len(chars)), dtype=np.bool)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_indices[char]] = True
    y[i, char_indices[next_chars[i]]] = True

# Build RNN model
model = Sequential()
model.add(LSTM(128, input_shape=(maxlen, len(chars))))
model.add(Dense(len(chars), activation='softmax'))

model.compile(optimizer='rmsprop', loss='categorical_crossentropy')

# Train model
model.fit(X, y, batch_size=128, epochs=10)

# Text generation
def generate_text(seed_text, next_chars=100):
    generated = ''
    sentence = seed_text.lower()
    for _ in range(next_chars):
        x_pred = np.zeros((1, maxlen, len(chars)))
        for t, char in enumerate(sentence):
            x_pred[0, t, char_indices[char]] = 1
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = np.argmax(preds)
        next_char = indices_char[next_index]
        generated += next_char
        sentence = sentence[1:] + next_char
    return generated

print(generate_text("The quick"))


In [ ]:
# 3. Perform sentiment analysis using a simple CNN model

import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Conv1D, GlobalMaxPooling1D, Embedding
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

# Sample data
texts = ["I love natural language processing", "Deep learning is amazing", "Keras makes life easy"]
labels = [1, 1, 0]

# Tokenization
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index

data = pad_sequences(sequences, maxlen=10)

# Build CNN model
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=32, input_length=10))
model.add(Conv1D(32, 3, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(data, np.array(labels), epochs=10, batch_size=2)


In [ ]:
# 4. Perform Named Entity Recognition (NER) using spaCy

import spacy

nlp = spacy.load("en_core_web_sm")
text = "Barack Obama was the 44th President of the United States."
doc = nlp(text)
for ent in doc.ents:
    print(ent.text, ent.label_)


In [ ]:
# 5. Implement a simple Seq2Seq model for machine translation using LSTM in Keras

import numpy as np
from keras.models import Model
from keras.layers import Input, LSTM, Dense
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

# Sample data
input_texts = ["Hello", "How are you?", "Good morning"]
target_texts = ["Hola", "¿Cómo estás?", "Buenos días"]

# Tokenization
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(input_texts + target_texts)
input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)
input_sequences = pad_sequences(input_sequences, maxlen=10)
target_sequences = pad_sequences(target_sequences, maxlen=10)

# Model parameters
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(input_dim=10000, output_dim=latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(input_dim=10000, output_dim=latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(10000, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy')

# Prepare target data for training
target_sequences = np.expand_dims(target_sequences, -1)
target_data = np.zeros((len(target_texts), 10, 10000))
for i, seq in enumerate(target_sequences):
    for t, word_id in enumerate(seq):
        if word_id > 0:
            target_data[i, t, word_id[0]] = 1

# Train model
model.fit([input_sequences, target_sequences], target_data, epochs=10, batch_size=2)


In [ ]:
# 6. Generate text using a pre-trained transformer model (GPT-2)

from transformers import pipeline

generator = pipeline("text-generation", model="gpt-2")
result = generator("Once upon a time", max_length=50, num_return_sequences=1)
print(result)


In [ ]:
# 7. Apply data augmentation for text in NLP

from nlpaug.augmenter.word import SynonymAug

# Sample data
text = "Natural Language Processing with Python is fun and interesting."

# Data augmentation using synonyms
aug = SynonymAug(aug_src='wordnet')
augmented_text = aug.augment(text)
print(augmented_text)


In [ ]:
# 8. Add an Attention Mechanism to a Seq2Seq model

import numpy as np
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Embedding, dot, Activation, concatenate
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

# Sample data
input_texts = ["Hello", "How are you?", "Good morning"]
target_texts = ["Hola", "¿Cómo estás?", "Buenos días"]

# Tokenization
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(input_texts + target_texts)
input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)
input_sequences = pad_sequences(input_sequences, maxlen=10)
target_sequences = pad_sequences(target_sequences, maxlen=10)

# Model parameters
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(input_dim=10000, output_dim=latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(input_dim=10000, output_dim=latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# Attention mechanism
attention = dot([decoder_outputs, encoder_outputs], axes=[2, 2])
attention = Activation('softmax')(attention)
context = dot([attention, encoder_outputs], axes=[2, 1])
decoder_combined_context = concatenate([context, decoder_outputs])

# Output layer
[_{{{CITATION{{{_1{](https://github.com/wzard/Algora/tree/7f827c94ec778ac189031c2faf02b2b15b966768/datasets%2Fbase.py)[_{{{CITATION{{{_2{](https://github.com/ericrincon/DeepLANBot/tree/435e1213cebb33f3e9bd15e3bf350c301dbc5068/Experiment.py)[_{{{CITATION{{{_3{](https://github.com/hosankang/AI_Class/tree/2fcac1e368b27c13fb4a4278c64ba63770d3d552/chars_rnn.py)